MixLoRA vs LoRA on Spivavtor GEC

In [1]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
def secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return None

for name in ["WANDB_API_KEY", "HF_TOKEN"]:
    if value := secret(name):
        os.environ[name] = value

REPO_DIR = "/kaggle/working/ai500-labs"
BRANCH = "main"
token = secret("GITHUB_TOKEN")
url = f"https://{token}@github.com/PavloKhomenko/ai500-labs.git" if token else "https://github.com/PavloKhomenko/ai500-labs.git"

In [2]:
# the token is passed only on the command line, origin keeps the public url (/kaggle/working gets saved)
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q {url} {BRANCH}
else:
    !git clone -q -b {BRANCH} {url} {REPO_DIR}
    !git -C {REPO_DIR} remote set-url origin https://github.com/PavloKhomenko/ai500-labs.git
!git -C {REPO_DIR} log --oneline -1

c824dd7 (HEAD -> main, origin/main, origin/HEAD) part 3 setup


In [3]:
# versions from lab-1/uv.lock; torch stays Kaggle's CUDA build
!pip install -q "transformers==5.18.0" "datasets==5.1.0" "accelerate==1.15.0" "peft==0.21.2" \
    "huggingface-hub==1.33.0" "wandb==0.30.0" "tabulate==0.10.0" "pyyaml==6.0.3" "sacrebleu==2.6.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.1/26.1 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import sys
sys.path.insert(0, f"{REPO_DIR}/lab-1/src")

import pandas as pd
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

from lab1.common import load_config, out_dir
from lab1.part2 import wandb_log as wl
from lab1.part3.mixlora import Adapter
from lab1.part3.sft import Part3Report, SFTTrainer, SpivavtorData
%matplotlib inline

cfg = load_config()
p3 = cfg["part3"]
device = torch.device("cuda")
env = {"torch": torch.__version__, "transformers": transformers.__version__, "gpu": torch.cuda.get_device_name()}
print(env)
p3_dir = out_dir(cfg, "part3")

def start_run(name, job_type, **extra):
    return wl.init_run(cfg, name, job_type, config={**p3, **env, **extra}, group="part3",
                       project=cfg["wandb"]["part3_project"])

def load_base():
    return AutoModelForCausalLM.from_pretrained(p3["model"], dtype=torch.float16).to(device)

{'torch': '2.11.0+cu128', 'transformers': '5.18.0', 'gpu': 'Tesla T4'}


In [5]:
tok = AutoTokenizer.from_pretrained(p3["model"])
data = SpivavtorData(tok, p3, seed=cfg["seed"])
data.check_masking()
data.check_disjoint()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

train.jsonl: downloading bytes:           |  0.00B            

val.jsonl:   0%|          | 0.00/3.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/62784 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6976 [00:00<?, ? examples/s]

{'train': 4000, 'val': 300, 'test': 300}
masking ok: 'І завжди у мене була якась “велика ціль”, на яку я відкладав гроші протягом місяців, а іноді навіть років.<|im_end|>'
splits disjoint: {'train': 4000, 'val': 300, 'test': 300}


In [6]:
!pip uninstall -q -y torchao

In [7]:
# checks in fp32 (fresh model each time, no deepcopy: T4 memory)
def load_fp32():
    return load_base().float().eval()

ids = torch.tensor([data.val[0]["ids"]], device=device)
base = load_fp32()
with torch.no_grad():
    ref = base(input_ids=ids).logits.float()
del base

checks = {}
for name, make in {"mixlora-none": lambda m: Adapter.mixlora(m, p3, "none"),
                   "mixlora-bias": lambda m: Adapter.mixlora(m, p3, "bias"),
                   "lora": lambda m: Adapter.lora(m, p3["lora_rank"])}.items():
    ad = make(load_fp32())
    ad.check_param_count()
    ad.check_identity(ids, ref)
    ad.check_roundtrip(ids, p3_dir / "roundtrip.pt")  # overwrites the weights: throwaway adapter
    checks[name] = ad.param_table()
    del ad
    torch.cuda.empty_cache()

params = checks["mixlora-none"]
RUNS = {
    "mixlora-none": lambda m: Adapter.mixlora(m, p3, "none"),
    "mixlora-aux": lambda m: Adapter.mixlora(m, p3, "aux"),
    "mixlora-bias": lambda m: Adapter.mixlora(m, p3, "bias"),
    f"lora-r{p3['lora_rank']}": lambda m: Adapter.lora(m, p3["lora_rank"]),
    f"lora-r{params['lora_rank_matched_trainable']}": lambda m: Adapter.lora(m, params["lora_rank_matched_trainable"]),
    f"lora-r{params['lora_rank_matched_active']}": lambda m: Adapter.lora(m, params["lora_rank_matched_active"]),
}
pd.DataFrame(checks).T

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable 117,555,200 == expected, active per token 32,915,456
identity: max|diff| = 1.64e-04
save/load round-trip: exact


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable 18,464,768 == expected, active per token 18,464,768
identity: max|diff| = 0.00e+00
save/load round-trip: exact


,trainable,active,lora_rank_matched_trainable,lora_rank_matched_active
mixlora-none,117555200,32915456,102,29
mixlora-bias,117555200,32915456,102,29
lora,18464768,18464768,16,16


In [8]:
for name in ["mixlora-none", f"lora-r{params['lora_rank_matched_trainable']}"]:
    trainer = SFTTrainer(RUNS[name](load_base()), data, p3, seed=cfg["seed"])
    print(name, trainer.train(out_dir(cfg, "part3", "timing", name), max_steps=8))
    del trainer
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{"loss": 0.2625, "aux_per_layer": 1.268, "grad_norm": 1.198, "step": 1, "lr": 2e-05, "sec_per_step": 5.835, "max_mem_gb": 7.018, "maxvio_mean": 1.39}
{"loss": 0.475, "aux_per_layer": 1.256, "grad_norm": 0.8622, "step": 8, "lr": 9e-05, "sec_per_step": 4.243, "max_mem_gb": 7.485, "maxvio_mean": 1.353}
mixlora-none {'steps': 8, 'sec_per_step': 4.100810209910075, 'peak_mem_gb': 7.484743118286133}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{"loss": 0.2625, "aux_per_layer": 0.0, "grad_norm": 3.568, "step": 1, "lr": 2e-05, "sec_per_step": 2.524, "max_mem_gb": 5.176}
{"loss": 0.4419, "aux_per_layer": 0.0, "grad_norm": 1.84, "step": 8, "lr": 9e-05, "sec_per_step": 2.177, "max_mem_gb": 5.494}
lora-r102 {'steps': 8, 'sec_per_step': 2.2386882305145264, 'peak_mem_gb': 5.494174480438232}


In [9]:
start_run("base", "eval")
trainer = SFTTrainer(Adapter.lora(load_base(), p3["lora_rank"]), data, p3, seed=cfg["seed"])
base_metrics = trainer.evaluate(out_dir(cfg, "part3", "base"), {"trainable": 0, "active": 0})
wl.finish()
del trainer
torch.cuda.empty_cache()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: pavlokhomenko2307 (ai340-nlp) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.30.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261013_150030-iqc3hqpo
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run base
wandb: ⭐️ View project at https://wandb.ai/ai340-nlp/ai500-lab1-part3
wandb: 🚀 View run at https://wandb.ai/ai340-nlp/ai500-lab1-part3/runs/iqc3hqpo


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

wandb: updating run metadata


{'val_loss': 0.4033941819560515, 'chrf': 55.45697659952204, 'exact_match': 0.0033333333333333335, 'trainable': 0, 'active': 0}


wandb: 
wandb: Run summary:
wandb:      active 0
wandb:        chrf 55.45698
wandb: exact_match 0.00333
wandb:   trainable 0
wandb:    val_loss 0.40339
wandb: 
wandb: 🚀 View run base at: https://wandb.ai/ai340-nlp/ai500-lab1-part3/runs/iqc3hqpo
wandb: ⭐️ View project at: https://wandb.ai/ai340-nlp/ai500-lab1-part3
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20261013_150030-iqc3hqpo/logs


In [10]:
for name, make in RUNS.items():
    run_dir = out_dir(cfg, "part3", name)
    if (run_dir / "metrics.json").exists():
        continue
    start_run(name, "train", run=name)
    trainer = SFTTrainer(make(load_base()), data, p3, seed=cfg["seed"])
    timing = trainer.train(run_dir)
    trainer.adapter.save(run_dir / "adapter.pt")
    trainer.evaluate(run_dir, timing)
    wl.finish()
    del trainer
    torch.cuda.empty_cache()

wandb: setting up run zvj9615p
wandb: Tracking run with wandb version 0.30.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261013_150446-zvj9615p
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run mixlora-none
wandb: ⭐️ View project at https://wandb.ai/ai340-nlp/ai500-lab1-part3
wandb: 🚀 View run at https://wandb.ai/ai340-nlp/ai500-lab1-part3/runs/zvj9615p


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{"loss": 0.2626, "aux_per_layer": 1.22, "grad_norm": 1.208, "step": 1, "lr": 2e-05, "sec_per_step": 4.96, "max_mem_gb": 7.002, "maxvio_mean": 1.218}
{"loss": 0.3451, "aux_per_layer": 1.211, "grad_norm": 0.7347, "step": 11, "lr": 0.00012, "sec_per_step": 4.137, "max_mem_gb": 7.482, "maxvio_mean": 1.219}
{"loss": 0.2519, "aux_per_layer": 1.211, "grad_norm": 0.4439, "step": 21, "lr": 0.0002, "sec_per_step": 4.212, "max_mem_gb": 7.61, "maxvio_mean": 1.142}
{"loss": 0.2595, "aux_per_layer": 1.239, "grad_norm": 0.4329, "step": 31, "lr": 0.000199, "sec_per_step": 4.689, "max_mem_gb": 7.61, "maxvio_mean": 1.298}
{"loss": 0.3324, "aux_per_layer": 1.248, "grad_norm": 0.4711, "step": 41, "lr": 0.0001963, "sec_per_step": 4.273, "max_mem_gb": 7.61, "maxvio_mean": 1.287}
{"loss": 0.3506, "aux_per_layer": 1.288, "grad_norm": 0.7066, "step": 51, "lr": 0.0001921, "sec_per_step": 4.755, "max_mem_gb": 7.61, "maxvio_mean": 1.432}
{"loss": 0.3664, "aux_per_layer": 1.329, "grad_norm": 0.5129, "step": 61, "l

wandb: updating run metadata


{'val_loss': 0.17527449178992285, 'chrf': 92.99179173286699, 'exact_match': 0.42, 'trainable': 117555200, 'active': 32915456, 'steps': 250, 'sec_per_step': 4.374765920639038, 'peak_mem_gb': 7.643589496612549}


...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{"loss": 0.2625, "aux_per_layer": 0.0, "grad_norm": 1.907, "step": 1, "lr": 2e-05, "sec_per_step": 2.765, "max_mem_gb": 3.858}
{"loss": 0.3182, "aux_per_layer": 0.0, "grad_norm": 0.9323, "step": 11, "lr": 0.00012, "sec_per_step": 2.426, "max_mem_gb": 4.209}
{"loss": 0.2554, "aux_per_layer": 0.0, "grad_norm": 0.6406, "step": 21, "lr": 0.0002, "sec_per_step": 2.477, "max_mem_gb": 4.311}
{"loss": 0.2607, "aux_per_layer": 0.0, "grad_norm": 0.6178, "step": 31, "lr": 0.000199, "sec_per_step": 2.59, "max_mem_gb": 4.445}
{"loss": 0.3294, "aux_per_layer": 0.0, "grad_norm": 0.6016, "step": 41, "lr": 0.0001963, "sec_per_step": 2.473, "max_mem_gb": 4.445}
{"loss": 0.3455, "aux_per_layer": 0.0, "grad_norm": 0.941, "step": 51, "lr": 0.0001921, "sec_per_step": 2.899, "max_mem_gb": 4.445}
{"loss": 0.3774, "aux_per_layer": 0.0, "grad_norm": 0.6837, "step": 61, "lr": 0.0001863, "sec_per_step": 2.7, "max_mem_gb": 4.445}
{"loss": 0.2524, "aux_per_layer": 0.0, "grad_norm": 0.4917, "step": 71, "lr": 0.00017

wandb: updating run metadata


{'val_loss': 0.1750112412135504, 'chrf': 92.78671822278308, 'exact_match': 0.42, 'trainable': 33467392, 'active': 33467392, 'steps': 250, 'sec_per_step': 2.4746897746105585, 'peak_mem_gb': 4.565489292144775}


wandb: uploading history steps 249-249, summary, console lines 26-26
wandb: 
wandb: Run history:
wandb: train/aux_per_layer ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:     train/grad_norm ██▁▅▄▆▃▁▄▂▄▄▁▃▄▄▂▁▅▆▄▄▂▄▃▃▅▇▃▁▂▅▆▅▃▄▂█▁▄
wandb:          train/loss ▆█▅▆▆▄▆▄▆▄▇▂▄▂▄▃▄▃▄▃▂▃▃▃▆▄▃▆▄▅▅▂▂▄▆▃▄▁▅▄
wandb:            train/lr ▂▅████████▇▇▇▇▇▇▇▆▆▆▆▆▆▆▆▅▅▄▄▃▂▂▂▂▂▁▁▁▁▁
wandb:    train/max_mem_gb ▁▄▅▅▅▇▇▇▇▇▇▇▇▇██████████████████████████
wandb:  train/sec_per_step ▂▂▅▂▃▅▂▃▂▄▄▃▅▅▄▂▇▃▅▆▃▃▆▅▇█▄▁▄▅▁▅▅▄▆▄█▇▄▃
wandb: 
wandb: Run summary:
wandb:              active 33467392
wandb:                chrf 92.78672
wandb:         exact_match 0.42
wandb:         peak_mem_gb 4.56549
wandb:        sec_per_step 2.47469
wandb:               steps 250
wandb: train/aux_per_layer 0
wandb:     train/grad_norm 0.60343
wandb:          train/loss 0.22734
wandb:            train/lr 2e-05
wandb:                  +4 ...
wandb: 
wandb: 🚀 View run lora-r29 at: https://wandb.ai/ai340-nlp/ai500-lab1-part3/runs/s8v5u3bq
wa

In [11]:
report = Part3Report(p3_dir, ["base", *RUNS])
start_run("summary", "analysis")
table = report.table()
report.routing_figures()
report.loss_curves()
wl.finish()
table

wandb: setting up run g1hmd2q2
wandb: Tracking run with wandb version 0.30.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261013_165745-g1hmd2q2
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run summary
wandb: ⭐️ View project at https://wandb.ai/ai340-nlp/ai500-lab1-part3
wandb: 🚀 View run at https://wandb.ai/ai340-nlp/ai500-lab1-part3/runs/g1hmd2q2
wandb: updating run metadata
wandb: uploading history steps 0-5, summary
wandb: 🚀 View run summary at: https://wandb.ai/ai340-nlp/ai500-lab1-part3/runs/g1hmd2q2
wandb: ⭐️ View project at: https://wandb.ai/ai340-nlp/ai500-lab1-part3
wandb: Synced 4 W&B file(s), 6 media file(s), 2 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20261013_165745-g1hmd2q2/logs


,run,chrF,EM,val_loss,trainable_M,active_M,sec_per_step,peak_mem_gb,maxvio_mean
0,base,55.456977,0.003333,0.403394,0.000000,0.000000,NaN,NaN,NaN
1,mixlora-none,92.991792,0.420000,0.175274,117.555200,32.915456,4.374766,7.643589,1.736614
2,mixlora-aux,92.551749,0.413333,0.174678,117.555200,32.915456,4.288287,7.616358,0.054328
3,mixlora-bias,92.828058,0.430000,0.175275,117.555200,32.915456,4.330522,7.613243,0.947275
4,lora-r16,92.669192,0.413333,0.177333,18.464768,18.464768,2.362030,4.341933,NaN
5,lora-r102,92.924723,0.436667,0.170927,117.712896,117.712896,2.589542,5.829476,NaN
6,lora-r29,92.786718,0.420000,0.175011,33.467392,33.467392,2.474690,4.565489,NaN
